# Module 2 — NLU & Entity Resolution (exercises)

You'll turn the repo's nickname+fuzzy `resolve_player` into an embedding-style linker and
**measure** the upgrade on a held-out eval set. Core cells are dependency-light (stdlib only).
The sentence-transformers cell is **OPTIONAL**.

Companion reading: `workbook.md`. Ship target: `src/statlas/entities/resolver.py`.

In [ ]:
import csv, math
from pathlib import Path

# Locate the repo root by walking up until we find src/statlas/data/seed/players.csv
here = Path.cwd()
root = None
for p in [here, *here.parents]:
    if (p / 'src/statlas/data/seed/players.csv').exists():
        root = p; break
assert root is not None, 'Run this notebook from inside the statlas repo.'
players_csv = root / 'src/statlas/data/seed/players.csv'

with open(players_csv) as f:
    PLAYERS = [{'player_id': int(r['player_id']), 'full_name': r['full_name']} for r in csv.DictReader(f)]
NAMES = [p['full_name'] for p in PLAYERS]
print('Loaded players:', NAMES)

# Nicknames: reuse the repo's dict (only depends on rapidfuzz which we won't call here).
try:
    import sys; sys.path.insert(0, str(root / 'src'))
    from statlas.entities.resolver import NICKNAMES
except Exception:
    NICKNAMES = {'wemby': 'Victor Wembanyama', 'lebron': 'LeBron James', 'bron': 'LeBron James',
                 'the joker': 'Nikola Jokic', 'joker': 'Nikola Jokic', 'kd': 'Kevin Durant',
                 'steph': 'Stephen Curry', 'chef curry': 'Stephen Curry'}
print('Nicknames:', len(NICKNAMES))

## Exercise 1 — The baseline
Here is a stdlib reimplementation of the repo's strategy (nickname hit → fuzzy ratio).
The repo uses `rapidfuzz.WRatio`; we use `difflib` so this runs with zero installs. Same shape.
Read it, then move on — it's your baseline to beat.

In [ ]:
from difflib import SequenceMatcher

def baseline_resolve(text, min_score=0.6):
    if not text: return None
    key = text.strip().lower()
    canon = NICKNAMES.get(key)
    if canon and canon in NAMES:
        return (canon, 1.0)
    best, best_s = None, 0.0
    for name in NAMES:
        s = SequenceMatcher(None, key, name.lower()).ratio()
        if s > best_s:
            best, best_s = name, s
    return (best, best_s) if best_s >= min_score else None

print(baseline_resolve('wemby'))
print(baseline_resolve('lebron james'))
print(baseline_resolve('the joker'))
print(baseline_resolve('Kawhi Leonard'))  # not in our 5-player DB -> should be None

## Exercise 2 — Build the eval set
A resolver is only as honest as its test set. Below is a starter of hard cases. **Add at least
8 more** of your own — typos, partial names, ambiguous refs, and a few that *should* be `None`
(out-of-DB). `None` means 'correct answer is: refuse to guess'.

In [ ]:
# (mention, expected_full_name_or_None)
EVAL = [
    ('wemby', 'Victor Wembanyama'),
    ('the alien', 'Victor Wembanyama'),     # nickname not in dict -> tail case
    ('lebron', 'LeBron James'),
    ('lebron jame', 'LeBron James'),        # typo
    ('LeBraun James', 'LeBron James'),      # misspelling
    ('the joker', 'Nikola Jokic'),
    ('jokic', 'Nikola Jokic'),
    ('steph curry', 'Stephen Curry'),
    ('durant', 'Kevin Durant'),
    ('KD', 'Kevin Durant'),
    ('Kawhi Leonard', None),                # not in DB
    ('Michael Jordan', None),               # not in DB
    # TODO: add >= 8 more cases (typos, partials, ambiguous, more None's).
]
assert len(EVAL) >= 20, f'extend EVAL to >= 20 cases (currently {len(EVAL)})'
assert any(exp is None for _, exp in EVAL), 'include out-of-DB cases that should resolve to None'
print('Eval cases:', len(EVAL))

## Exercise 3 — Score the baseline
Write a scorer: a prediction is correct if the resolved name equals the expected name, OR both
are `None`. Print accuracy and log the misses.

In [ ]:
def score(resolver, eval_set, **kwargs):
    # TODO: for each (mention, expected): pred = resolver(mention, **kwargs); pred_name = pred[0] if pred else None
    #       correct if pred_name == expected. Return (accuracy, misses) where misses is a list of
    #       (mention, expected, pred_name).
    raise NotImplementedError

acc, misses = score(baseline_resolve, EVAL)
print(f'Baseline accuracy: {acc:.1%}')
for m in misses: print('  miss:', m)
assert 0.0 <= acc <= 1.0

## Exercise 4 — Char-ngram cosine resolver (no deps)
Represent each name as a TF vector over character 3-grams; resolve a mention to the highest-cosine
name. This is a poor-man's embedding and often crushes edit-distance on misspellings.
Keep the nickname shortcut first.

In [ ]:
from collections import Counter

def char_ngrams(s, n=3):
    s = f'  {s.lower().strip()}  '
    # TODO: return a Counter of all length-n substrings of s.
    raise NotImplementedError

def cosine(a, b):
    # TODO: cosine similarity of two Counters. dot = sum a[k]*b[k]; norms = sqrt(sum v^2).
    raise NotImplementedError

_NAME_VECS = {name: char_ngrams(name) for name in NAMES}

def ngram_resolve(text, min_score=0.3):
    if not text: return None
    key = text.strip().lower()
    canon = NICKNAMES.get(key)
    if canon and canon in NAMES:
        return (canon, 1.0)
    qv = char_ngrams(key)
    best, best_s = None, 0.0
    for name, nv in _NAME_VECS.items():
        s = cosine(qv, nv)
        if s > best_s: best, best_s = name, s
    return (best, best_s) if best_s >= min_score else None

# --- tests ---
assert abs(cosine(char_ngrams('abc'), char_ngrams('abc')) - 1.0) < 1e-9, 'identical -> 1.0'
assert ngram_resolve('LeBraun James')[0] == 'LeBron James', 'should handle the misspelling'
acc_ng, misses_ng = score(ngram_resolve, EVAL)
print(f'N-gram accuracy: {acc_ng:.1%}  (baseline was scored above)')
for m in misses_ng: print('  miss:', m)

## Exercise 5 — OPTIONAL: sentence-embedding resolver
`pip install sentence-transformers`. Encode names + mention with `all-MiniLM-L6-v2`, cosine-rank.
On a 5-name DB this is overkill, but it's the pattern that scales to thousands of entities.
Score it on EVAL and compare.

In [ ]:
# OPTIONAL
# from sentence_transformers import SentenceTransformer, util
# model = SentenceTransformer('all-MiniLM-L6-v2')
# name_emb = model.encode(NAMES, convert_to_tensor=True)
# def st_resolve(text, min_score=0.4):
#     ... encode text, cosine vs name_emb, argmax, threshold ...
# print(score(st_resolve, EVAL)[0])
print('Optional — implement if sentence-transformers is installed.')

## Exercise 6 — Threshold sweep (precision vs refusal)
For a stats engine, a wrong-but-confident answer is worse than an honest 'I don't know'.
Sweep `min_score` for your n-gram resolver and watch accuracy vs how often it refuses (returns None).
Pick an operating point and justify it in `notes.md`.

In [ ]:
def refusal_rate(resolver, eval_set, **kwargs):
    return sum(1 for m, _ in eval_set if resolver(m, **kwargs) is None) / len(eval_set)

for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    a, _ = score(ngram_resolve, EVAL, min_score=t)
    r = refusal_rate(ngram_resolve, EVAL, min_score=t)
    bar = '#' * int(a * 30)
    print(f'min_score={t:.1f}  acc={a:5.1%}  refuse={r:5.1%}  |{bar}')

## Exercise 7 — Ship it
Port the winning resolver into `src/statlas/entities/resolver.py`:
- Keep the exact nickname lookup first (it's perfect for the head of the distribution).
- Add your n-gram (or embedding) path as the fuzzy fallback, **keeping** the
  `ResolvedPlayer(player_id, full_name, score)` return contract so `parser.py` is untouched.
- Leave the rapidfuzz path reachable behind a flag so M7's eval can A/B them.

Then run the repo's CLI on a few questions and confirm nothing downstream broke.
**Save your `EVAL` list** — it feeds the Module 7 evaluation harness.